In [ ]:
import xarray as xr
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import cmocean.cm as cmo
from matplotlib.gridspec import GridSpec, GridSpecFromSubplotSpec
from matplotlib import rc
from cartopy.crs import Mercator, PlateCarree

from plot import nice_lonlat_gridlines, scale_bar
from pinnacle45 import beam_direction_instrument__Pin45, roation_matrix_instrument2earth__Pin45

rc('font', size=6)
plt.rcParams['font.sans-serif'] = ['Arial'] + plt.rcParams['font.sans-serif'] # Arial as first choice

fig_width = 5.5 
fig_height = 4
cbar_aspect = 20
cbar_pad = 0.02

### ADCP-derived maps

In [ ]:
lon_min = -113.5
lon_max = -112
lat_max = -74.05
lat_min = -74.3

vmin=0
vmax=360
cmap=cmo.haline
step=1

proj = Mercator(central_longitude=-112.75,
                min_latitude = -75,
                max_latitude = -73,
                latitude_true_scale = -74.2)

# Plot maps
corner = [[-113.4, -74.16],
          [-112.65, -74.21]]
width  = 0.6 # degrees lon
height = 0.1 #degrees north
lon_grid = [[-113.3, -113.1, -112.9],
            [-112.5, -112.3, -112.1]]
lat_grid = [[-74.18, -74.23],
            [-74.23, -74.28]]

In [ ]:
labels = [f'NBP2202_0{i}' for i in [2,3,4]]
files_adcp = [f'data/derived/{label}_ice_draft.csv' for label in labels]
ice_drafts = [pd.read_csv(f) for f in files_adcp]
sar = xr.load_dataset('data/auxiliary/background/2022-01-21-00_00_2022-01-21-23_59_Sentinel-1_IW_HH_HH_-_decibel_gamma0.nc')

### Beams

In [ ]:
beams = [1,2,3,4]
beam_colors = {1: 'tab:blue', 2: 'tab:red', 3: 'black', 4: 'tab:orange'}
beam_labels = {1: 'starboard', 2: 'port', 3: 'fore', 4: 'aft'}

### Side-view

In [ ]:
imin = 10180
imax = 10990
index = range(imin,imax)

side_view = {}
for b in beams:
    side_view[b] = ice_drafts[2].where(ice_drafts[2].beam==b).loc[imin:imax].dropna()

auv_depth = ice_drafts[2].loc[imin:imax].AUV_depth
auv_lat   = ice_drafts[2].loc[imin:imax].AUV_lat

### Beam angle

In [ ]:
orientation_data = [xr.open_dataset(f'data/derived/NBP2202_0{i}_cleaned.nc') for i in [2,3,4]]

for (i,ds) in enumerate(orientation_data):
    beam_direction__inst = beam_direction_instrument__Pin45()
    inst2earth = roation_matrix_instrument2earth__Pin45(ds,
                                                        pitch='pitch_adcp', 
                                                        roll='roll_adcp', 
                                                        heading='heading_adcp', 
                                                        time_dim='time')
    ds['beam_direction__earth'] = inst2earth @ beam_direction__inst 
    orientation_data[i]['theta0'] = np.degrees(np.arcsin(ds.beam_direction__earth.sel(earth='U')))

thetas = {}
for b in beams:
    theta = []
    for ds in orientation_data:
        theta.append(ds.theta0.sel(beam=b).values)
    thetas[b] = np.concatenate(theta)

### Figure

In [ ]:
fig = plt.figure(figsize=(fig_width, fig_height))

gs0 = GridSpec(1, 2, figure=fig, width_ratios = [1.2,1],wspace=0.03)

gs00 = GridSpecFromSubplotSpec(2,3, subplot_spec=gs0[0], height_ratios=(2,0.07), width_ratios = (0.03,1,0.05), hspace=0.15)
gs10 = GridSpecFromSubplotSpec(2,1, subplot_spec=gs00[0,:], hspace= 0.05)
axa = fig.add_subplot(gs10[0], projection = proj)
axb = fig.add_subplot(gs10[1], projection = proj)
cax = fig.add_subplot(gs00[1,1])

gs01 = gs0[1].subgridspec(2, 1, height_ratios = [1.8, 1], hspace=0.05)
axc = fig.add_subplot(gs01[0, :])
axd = fig.add_subplot(gs01[1, :])

# ---- Maps ----
step=1
for (ax,co, lons, lats, tick_pos) in zip([axa,axb],corner, lon_grid, lat_grid, [['top', 'left'], ['bottom', 'left']]):   
    ax.pcolormesh(sar.lon, sar.lat, sar.Band1, cmap=cmo.gray, transform = PlateCarree(), zorder=-10)
    for map in ice_drafts:
        im = ax.scatter(map.longitude[::step], map.latitude[::step], transform=PlateCarree(), 
                        s=1, marker='.',
                        c=map.ice_draft[::step], vmin=vmin, vmax=vmax, cmap=cmap)
    ax.set_extent([co[0], co[0]+width, co[1]-height, co[1]], crs=PlateCarree())
    gl = nice_lonlat_gridlines(ax, size=6, zorder=-10, longitudes = lons, latitudes = lats, labels=tick_pos, alpha=0.2)
    scale_bar(ax, length=2, location = (0.03,0.03), textoffset=90, linewidth=2)

# Add colorbar
cbar = plt.colorbar(im, cax=cax, aspect = 30, extend='both', orientation='horizontal', pad=0)
cbar.ax.tick_params(labelsize=7, length=2)
cbar.set_label( label= 'Ice draft (m)')

# ---- Side view (panel c) ----
for b in [3,4]:
    axc.scatter(side_view[b].latitude, side_view[b].ice_draft, s=1, label=f'{beam_labels[b]} beam', color=beam_colors[b])
axc.scatter(auv_lat, auv_depth, c='gray',s=1 , label='AUV position')
axc.invert_yaxis()
axc.yaxis.tick_right()
axc.yaxis.set_label_position("right")
axc.xaxis.tick_top()
xlim = axc.get_xlim()
xticks = [-74.28, -74.26, -74.24]
xlabels = [f'{-x:1.2f}°S' for x in xticks]
axc.set_xticks(xticks, labels=xlabels)
axc.set_xlim(xlim)
axc.grid(alpha=0.1)
axc.set_ylabel('depth (m)')
axc.tick_params(length=0, pad=2)
lgnd = axc.legend(loc = 'center left')
# Increase size of scatter points in legend
for handle in lgnd.legend_handles[:]:
    handle.set_sizes([7.0])

# --- Theta histogram (panel d) ---
bins = np.linspace(10,90,81)
for b in beams:
    axd.hist(thetas[b], bins=bins, label=f'{beam_labels[b]} beam', alpha=0.5, color=beam_colors[b])
    axd.hist(thetas[b], bins=bins, histtype='step', color=beam_colors[b])
axd.legend()
axd.set_xlabel(r'$\theta_0$')
axd.set_ylabel('count')
axd.yaxis.tick_right()
axd.yaxis.set_label_position("right")
axd.tick_params(length=0, pad=2)
xticks = np.arange(20,91,10)
xlabels = [f'{x:.0f}°' for x in xticks]
axd.set_xticks(xticks, labels=xlabels)
axd.set_xlim([15,95])
#axd.set_ylim(0.9, 5e4)
#axd.set_yscale('log')
axd.grid('both',alpha=0.1, zorder=-10)

# --- Annotate panels---
axes = [axa, axb, axc, axd]
ax_labels = ['(a)', '(b)', '(c)', '(d)']
label_colors = ['w', 'w', 'k', 'k']
ys = [0.9, 0.9, 0.94, 0.88]
xs = [0.01, 0.01, 0.02, 0.92]
for (ax, label, c, x, y) in zip(axes, ax_labels, label_colors, xs, ys):
    ax.annotate(label, xy=(x, y), xycoords='axes fraction', color = c, weight='bold')

plt.savefig('figures/fig5.png', bbox_inches = 'tight', dpi=600)